In [ ]:
import os
import glob
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [24]:
def find_project_root(anchor="data") -> Path:
    """
    Detecta dinámicamente la raíz del proyecto buscando la carpeta ancla
    hacia arriba a partir del directorio actual o del archivo.
    """
    # En notebooks usamos Path.cwd(), en scripts usamos Path(__file__)
    current = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
    
    for candidate in [current, *current.parents]:
        if (candidate / anchor).exists():
            return candidate
            
    raise FileNotFoundError(f"No se pudo localizar la carpeta ancla '{anchor}' en la jerarquía superior.")


In [25]:
# 1. Raíz absoluta del proyecto 
PROJECT_ROOT = find_project_root(anchor="data")

# 2. Definición de rutas absolutas mediante el operador / de pathlib
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
GEO_DIR = RAW_DIR / "geographic"
ECONOMIC_DIR = RAW_DIR / "economic"

print(f" Raíz del proyecto detectada : {PROJECT_ROOT}")
print(f" Carpeta RAW confirmada      : {RAW_DIR} (Existe: {RAW_DIR.exists()})")

 Raíz del proyecto detectada : /Users/eli/Documents/UPY/9no/Business Intelligence/urban_intelligence
 Carpeta RAW confirmada      : /Users/eli/Documents/UPY/9no/Business Intelligence/urban_intelligence/data/raw (Existe: True)


In [26]:
# ==========================================================
# 1. VERIFICAR CAPA GEOGRÁFICA (SHAPEFILES INEGI)
# ==========================================================
print("\n[1] INSPECCIONANDO CAPAS GEOGRÁFICAS...")

shapefile_configs = {
    "AGEB Urbana": os.path.join(GEO_DIR, "2020_1_09_A", "2020_1_09_A.shp"),
    "Alcaldías (MUN)": os.path.join(GEO_DIR, "2020_1_09_MUN", "2020_1_09_MUN.shp"),
    "Entidad (ENT)": os.path.join(GEO_DIR, "2020_1_09_ENT", "2020_1_09_ENT.shp"),
}

for label, shp_path in shapefile_configs.items():
    if os.path.exists(shp_path):
        gdf = gpd.read_file(shp_path)
        print(f" -> {label}:")
        print(f"    - Archivo: {shp_path}")
        print(f"    - Registros/Polígonos: {len(gdf)}")
        print(f"    - CRS detectado: {gdf.crs}")
        print(f"    - Geometrías válidas: {gdf.is_valid.sum()} de {len(gdf)}")
        print(f"    - Columnas disponibles: {list(gdf.columns[:6])}...")
    else:
        # Búsqueda alternativa por si el nombre interno varía
        folder = os.path.dirname(shp_path)
        alt_shps = glob.glob(os.path.join(folder, "*.shp")) if os.path.exists(folder) else []
        if alt_shps:
            gdf = gpd.read_file(alt_shps[0])
            print(f" -> {label} (nombre alternativo):")
            print(f"    - Archivo: {alt_shps[0]}")
            print(f"    - Registros: {len(gdf)} | CRS: {gdf.crs}")
        else:
            print(f" -> {label}: [NO ENCONTRADO en {folder}]")


[1] INSPECCIONANDO CAPAS GEOGRÁFICAS...
 -> AGEB Urbana:
    - Archivo: /Users/eli/Documents/UPY/9no/Business Intelligence/urban_intelligence/data/raw/geographic/2020_1_09_A/2020_1_09_A.shp
    - Registros/Polígonos: 2431
    - CRS detectado: PROJCS["MEXICO_ITRF_2008_LCC",GEOGCS["ITRF2008",DATUM["International_Terrestrial_Reference_Frame_2008",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1061"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",12],PARAMETER["central_meridian",-102],PARAMETER["standard_parallel_1",17.5],PARAMETER["standard_parallel_2",29.5],PARAMETER["false_easting",2500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
    - Geometrías válidas: 2431 de 2431
    - Columnas disponibles: ['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'CVE_LOC', 'CVE_AGEB', 'geometry']...
 -> Alcaldías (MUN):
    

In [27]:
# ==========================================================
# 2. VERIFICAR ARCHIVO DEMOGRÁFICO (CENSO 2020)
# ==========================================================
print("\n[2] INSPECCIONANDO CENSO DE POBLACIÓN (POPULATION_DATASET.CSV)...")
pop_path = os.path.join(RAW_DIR, "population_dataset.csv")

if os.path.exists(pop_path):
    # Cargar solo las primeras filas para verificar lectura rápida
    df_pop_head = pd.read_csv(pop_path, nrows=5)
    print(f" -> Archivo cargado correctamente: {pop_path}")
    print(f" -> Total de columnas detectadas: {len(df_pop_head.columns)}")
    
    # Revisar presencia de variables clave
    cols_demograficas = ["POBTOT", "PEA", "P_15A64", "CVEGEO", "CVE_AGEB", "NOM_LOC", "MZA"]
    encontradas = [c for c in cols_demograficas if c in df_pop_head.columns]
    print(f" -> Columnas requeridas encontradas: {encontradas}")
    print(f" -> Muestra de columnas: {list(df_pop_head.columns[:8])}")
else:
    print(f" -> [ERROR] No se localizó {pop_path}")


[2] INSPECCIONANDO CENSO DE POBLACIÓN (POPULATION_DATASET.CSV)...
 -> Archivo cargado correctamente: /Users/eli/Documents/UPY/9no/Business Intelligence/urban_intelligence/data/raw/population_dataset.csv
 -> Total de columnas detectadas: 230
 -> Columnas requeridas encontradas: ['POBTOT', 'PEA', 'NOM_LOC', 'MZA']
 -> Muestra de columnas: ['ENTIDAD', 'NOM_ENT', 'MUN', 'NOM_MUN', 'LOC', 'NOM_LOC', 'AGEB', 'MZA']


In [28]:
# ==========================================================
# 3. VERIFICAR ARCHIVO DE DELITOS (CRIME_DATASET.CSV)
# ==========================================================
print("\n[3] INSPECCIONANDO DELITOS (CRIME_DATASET.CSV)...")
crime_path = os.path.join(RAW_DIR, "crime_dataset.csv")

if os.path.exists(crime_path):
    df_crime_head = pd.read_csv(crime_path, nrows=5)
    print(f" -> Archivo cargado correctamente: {crime_path}")
    print(f" -> Total de columnas detectadas: {len(df_crime_head.columns)}")
    
    cols_crime = [c.lower() for c in df_crime_head.columns]
    tiene_coords = "latitud" in cols_crime and "longitud" in cols_crime
    tiene_hecho = "anio_hecho" in cols_crime or "fecha_hecho" in cols_crime
    
    print(f" -> ¿Tiene coordenadas (latitud/longitud)?: {'SÍ' if tiene_coords else 'NO'}")
    print(f" -> ¿Tiene temporalidad (fecha/año hecho)?: {'SÍ' if tiene_hecho else 'NO'}")
    print(f" -> Muestra de columnas: {list(df_crime_head.columns[:8])}")
else:
    print(f" -> [ERROR] No se localizó {crime_path}")


[3] INSPECCIONANDO DELITOS (CRIME_DATASET.CSV)...
 -> Archivo cargado correctamente: /Users/eli/Documents/UPY/9no/Business Intelligence/urban_intelligence/data/raw/crime_dataset.csv
 -> Total de columnas detectadas: 22
 -> ¿Tiene coordenadas (latitud/longitud)?: SÍ
 -> ¿Tiene temporalidad (fecha/año hecho)?: SÍ
 -> Muestra de columnas: ['_id', 'anio_inicio', 'mes_inicio', 'fecha_inicio', 'hora_inicio', 'anio_hecho', 'mes_hecho', 'fecha_hecho']


In [29]:
# ==========================================================
# 4. VERIFICAR ACTIVIDAD ECONÓMICA (DENUE)
# ==========================================================
print("\n[4] INSPECCIONANDO DENUE...")
denue_csv_files = glob.glob(os.path.join(RAW_DIR, "economic", "**", "*.csv"), recursive=True) + \
                  glob.glob(os.path.join(RAW_DIR, "**", "denue*.csv"), recursive=True)

if denue_csv_files:
    denue_sample = pd.read_csv(denue_csv_files[0], nrows=5, encoding="latin1", low_memory=False)
    print(f" -> Archivo CSV detectado: {denue_csv_files[0]}")
    print(f" -> Columnas detectadas: {len(denue_sample.columns)}")
    denue_cols = [c.lower() for c in denue_sample.columns]
    tiene_scian = any("scian" in c or "act" in c for c in denue_cols)
    print(f" -> ¿Contiene clasificación económica / SCIAN?: {'SÍ' if tiene_scian else 'NO'}")
else:
    print(" -> No se encontraron CSVs de DENUE en subcarpetas de economic.")



[4] INSPECCIONANDO DENUE...
 -> Archivo CSV detectado: /Users/eli/Documents/UPY/9no/Business Intelligence/urban_intelligence/data/raw/economic/denue_09_csv/diccionario_de_datos/denue_diccionario_de_datos.csv
 -> Columnas detectadas: 5
 -> ¿Contiene clasificación económica / SCIAN?: NO
